# Huấn luyện mô hình quyết định kiểu Jev (NT532)

Notebook train hai mô hình song song rồi chấm trên test và test_shift:

- `jev1`: EmbeddingGemma 2 + LoRA + đầu Choice/Boolean, chạy trên GPU 0.
- `heads_only`: đối chứng, đóng băng backbone, chỉ train đầu, chạy trên GPU 1.

**Yêu cầu:** Accelerator GPU T4 x2, Internet ON, đã thêm Dataset chứa 4 file `train/val/test/test_shift.jsonl`
(xem `kaggle/README.md`). Chạy được ở chế độ Save & Run All (không cần tương tác).

**Kết quả:** `/kaggle/working/jev_results.zip` gồm `runs/decider/jev/jev1` và `heads_only`
(adapter LoRA, đầu, config, eval.json, log), không gồm mô hình gốc.

**Lưu ý:** model `google/embeddinggemma-2` có thể cần chấp nhận điều khoản trên Hugging Face; nếu tải lỗi 401/403
hãy thêm HF_TOKEN vào Kaggle Secrets.

**Chấm lại không train:** Add Data, chọn Notebook Output của phiên bản cũ (có `jev_results.zip`). Lượt nào đã có
`config.json` thì bỏ qua train và chỉ chấm điểm.

In [ ]:
!nvidia-smi
!git clone https://github.com/knight3007/Project-NT532.R12.git /kaggle/working/repo

## Cài thư viện
Dùng torch có sẵn của Kaggle (không cài lại). Chỉ thêm sentence-transformers, transformers>=5, peft và
cài gói `nt532` ở chế độ editable (không kéo thêm phụ thuộc). Phần quyết định chỉ cần numpy và pyyaml.

In [ ]:
# Kaggle cài sẵn transformers cũ (5.16 chưa biết embedding_gemma2) nên phải ép nâng cấp (-U).
!pip install -q -U "transformers>=5.19" "sentence-transformers>=6.1" "peft>=0.13" pyyaml psutil
!pip install -q -e /kaggle/working/repo/pi --no-deps
# torchao 0.10 có sẵn trên Kaggle làm peft mới báo lỗi khi gắn LoRA; mình không dùng nên gỡ.
!pip uninstall -y -q torchao
import torch, transformers, sentence_transformers
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| sentence-transformers", sentence_transformers.__version__, "| GPU:", torch.cuda.device_count())
ver = tuple(int(x) for x in transformers.__version__.split(".")[:2])
assert ver >= (5, 19), "cần transformers>=5.19 cho embedding_gemma2"
print("bf16 hỗ trợ:", torch.cuda.is_bf16_supported(), "(T4 không có bf16 gốc nên dùng fp32)")
# Thử nạp cấu hình model ngay, lỗi thì dừng ở đây thay vì sau 10 phút train.
from transformers import AutoConfig
print(AutoConfig.from_pretrained("google/embeddinggemma-2").model_type)

## Gắn dữ liệu
Tìm `train.jsonl` bất kỳ đâu dưới `/kaggle/input` rồi chép 4 file vào `<repo>/runs/decider/scenarios/`
(đường dẫn mà script mong đợi, vì `REPO_ROOT` là thư mục repo).

In [ ]:
import glob, os, shutil
hits = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)
assert hits, "không thấy train.jsonl dưới /kaggle/input; hãy Add Data với dataset nt532-scenarios"
src = os.path.dirname(hits[0])
dst = "/kaggle/working/repo/runs/decider/scenarios"
os.makedirs(dst, exist_ok=True)
for n in ("train", "val", "test", "test_shift"):
    shutil.copy(f"{src}/{n}.jsonl", f"{dst}/{n}.jsonl")
!ls -la {dst}

## Nạp kết quả cũ (nếu có) và kiểm tra đường chấm điểm
Tìm `jev_results.zip` hoặc thư mục `<lượt>/config.json` dưới `/kaggle/input` rồi chép vào `runs/decider/jev/`.
Sau đó chạy `eval_rules.py` (CPU, vài giây): lỗi ở bước chấm thì dừng ngay, không đợi tới sau khi train.

In [ ]:
import json, zipfile, subprocess
JEV = "/kaggle/working/repo/runs/decider/jev"
os.makedirs(JEV, exist_ok=True)
for z in glob.glob("/kaggle/input/**/jev_results.zip", recursive=True):
    print("giải nén", z)
    zipfile.ZipFile(z).extractall(JEV)
for cfg in glob.glob("/kaggle/input/**/config.json", recursive=True):
    run = os.path.dirname(cfg)
    name = os.path.basename(run)
    if os.path.exists(f"{run}/heads.pt") and not os.path.exists(f"{JEV}/{name}/config.json"):
        print("chép", run)
        shutil.copytree(run, f"{JEV}/{name}", dirs_exist_ok=True)
done = sorted(n for n in os.listdir(JEV) if os.path.exists(f"{JEV}/{n}/config.json")
              and "partial_step" not in json.load(open(f"{JEV}/{n}/config.json")))
print("đã có mô hình:", done)
r = subprocess.run(["python", "scripts/eval_rules.py"], cwd="/kaggle/working/repo/pi",
                   capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-3000:])
assert r.returncode == 0, "eval_rules.py lỗi, eval_jev.py cũng sẽ lỗi; sửa trước khi train"

## Chọn cấu hình
Đo cục bộ (RTX 4050 6 GB, fp32, batch 1): khoảng 1,25 bản ghi/s; T4 có fp32 cùng cỡ nên giả định 1,0-1,3 bản ghi/s.
Toàn bộ train (27,7k bản ghi) cần khoảng 6-7,5 giờ, quá sát giới hạn phiên 9 giờ khi còn phải chấm điểm,
nên dùng `--max-records 20000` (khoảng 4,5-5,5 giờ). `heads_only` cùng ngân sách bản ghi nhưng nhanh hơn (không lan truyền ngược qua backbone).
Hai tiến trình chạy song song, mỗi cái một GPU. Batch 4 x accum 4 = 16 bản ghi mỗi bước; peak VRAM fp32 batch 1 chỉ khoảng 2,6 GB nên
T4 16 GB dư sức. Có thể chỉnh `BATCH`, `ACCUM`, `MAX_RECORDS` bên dưới (đặt `MAX_RECORDS = 0` để dùng toàn bộ train).
Script dừng nếu RAM máy chủ vượt `--max-rss-gb` (ở đây 24 GB vì Kaggle có khoảng 30 GB).
Lưu ý: chỉ lưu kết quả ở cuối nên nếu phiên bị ngắt giữa chừng sẽ mất phần train đang chạy.

In [ ]:
import os, subprocess, time
REPO = "/kaggle/working/repo"
PI = f"{REPO}/pi"
LOGS = "/kaggle/working/logs"
os.makedirs(LOGS, exist_ok=True)
BATCH, ACCUM, MAX_RECORDS = 4, 4, 20000
# Lượt nào cần chạy; mỗi lượt một GPU theo thứ tự.
RUNS = ['jev1', 'heads_only']
common = ["--precision", "fp32", "--epochs", "1", "--batch", str(BATCH), "--accum", str(ACCUM),
          "--max-rss-gb", "24"]
if MAX_RECORDS:
    common += ["--max-records", str(MAX_RECORDS)]

def launch(name, gpu, extra=()):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
    cmd = ["python", "scripts/train_jev.py", "--device", "cuda", "--name", name, *common, *extra]
    log = open(f"{LOGS}/{name}.log", "w")
    return subprocess.Popen(cmd, cwd=PI, env=env, stdout=log, stderr=subprocess.STDOUT)

EXTRA = {"heads_only": ["--freeze-backbone"]}
TRAIN = [n for n in RUNS if n not in done]  # lượt đã có mô hình hoàn chỉnh thì chỉ chấm
print("train:", TRAIN, "| bỏ qua:", [n for n in RUNS if n in done])
procs = {name: launch(name, gpu, EXTRA.get(name, ())) for gpu, name in enumerate(TRAIN)}
print({k: p.pid for k, p in procs.items()})

## Theo dõi (đợi cả hai chạy xong; in 5 dòng cuối của mỗi log mỗi 10 phút)

In [ ]:
def tail(name, n=3, everything=False):
    with open(f"{LOGS}/{name}.log", errors="replace") as f:
        lines = [l.rstrip() for l in f]
    if not everything:
        lines = [l for l in lines if "bước" in l or "val loss" in l or "DỪNG" in l]
    return "\n".join(lines[-n:])

# Kiểm tra mỗi phút; in tiến độ mỗi 10 phút; tiến trình nào chết thì in 40 dòng cuối log của nó.
reported, t = set(), 0
while any(p.poll() is None for p in procs.values()):
    time.sleep(60)
    t += 1
    for k, p in procs.items():
        if p.poll() not in (None, 0) and k not in reported:
            reported.add(k)
            print(f"[{k}] LỖI, mã {p.returncode}:\n{tail(k, 40, everything=True)}\n")
    if t % 10 == 0:
        for k in procs:
            print(f"[{k}] {tail(k)}\n")
print({k: p.returncode for k, p in procs.items()})
for k, p in procs.items():
    if p.returncode != 0 and k not in reported:
        print(f"[{k}] LỖI:\n{tail(k, 40, everything=True)}")
assert all(p.returncode == 0 for p in procs.values()), "một lần train lỗi, xem log ở trên"

## Chấm điểm trên test và test_shift (song song hai GPU)

In [ ]:
evals = {}
for gpu, name in enumerate(RUNS):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
    out = f"{PI}/../runs/decider/jev/{name}/eval.log"
    f = open(out, "w")
    evals[name] = subprocess.Popen(
        ["python", "scripts/eval_jev.py", "--name", name, "--device", "cuda", "--precision", "fp32"],
        cwd=PI, env=env, stdout=f, stderr=subprocess.STDOUT)
for name, p in evals.items():
    p.wait()
    print(name, "exit", p.returncode)
for name in evals:
    print("=" * 30, name)
    print(open(f"{REPO}/runs/decider/jev/{name}/eval.log", errors="replace").read()[-6000:])
assert all(p.returncode == 0 for p in evals.values()), "chấm điểm lỗi, xem eval.log ở trên"

## Đóng gói kết quả

In [ ]:
import shutil, zipfile
for name in TRAIN:
    shutil.copy(f"{LOGS}/{name}.log", f"{REPO}/runs/decider/jev/{name}/train.log")
with zipfile.ZipFile("/kaggle/working/jev_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for name in RUNS:
        base = f"{REPO}/runs/decider/jev/{name}"
        for root, _, files in os.walk(base):
            for fn in files:
                path = os.path.join(root, fn)
                z.write(path, os.path.relpath(path, f"{REPO}/runs/decider/jev"))
    print(z.namelist())
!ls -la /kaggle/working/jev_results.zip